# MNIST MLP3 — Corrected Full Matrix-Log RG

Scientific hierarchy: `cone` + `projected_state` is the primary method; `radial` is the conservative reference; `modewise` and `post_step` are legacy ablations. The validation grid tests `full_m` versus `self_consistent` normalization without evaluating the official test set.


In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import json, os, sys
import pandas as pd

REPO = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p/'baseline'/'rg_baselines').is_dir() and (p/'optimizers'/'full_matrix_log_rg'/'full_matrix_log_rg').is_dir()), None)
if REPO is None:
    raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers')
for path in (REPO/'baseline', REPO/'optimizers'/'full_matrix_log_rg'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from rg_baselines import BaselineConfig, DEFAULT_BASELINE_SEEDS, summarize_numeric_metrics
from rg_baselines.engine import choose_device
from full_matrix_log_rg import FullMatrixLogConfig
from full_matrix_log_rg.experiment import run_mnist_sgd, run_validation_grid

DEVICE = choose_device()
DATA_DIR = Path(os.environ.get('RG_BASELINE_DATA_DIR', Path.home()/'rg-optimizer-data')).expanduser().resolve()
RUN_ROOT = Path(os.environ.get('RG_FML_RUN_ROOT', Path.home()/'rg-optimizer-runs'/'full_matrix_log_rg_corrected')).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True); RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({'device': str(DEVICE), 'data': str(DATA_DIR), 'runs': str(RUN_ROOT)})


In [ ]:
# Preflight: geometry, active-set KKT, normalization, projected momentum, restart.
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO/'optimizers'/'full_matrix_log_rg'/'tests'), '-v'], check=True, env={**os.environ, 'PYTHONPATH': str(REPO/'optimizers'/'full_matrix_log_rg')})


In [ ]:
BASE = BaselineConfig(optimizer='sgd_momentum', epochs=30, validation_size=5_000, sgd_learning_rate=0.05, sgd_min_learning_rate=5e-4, sgd_warmup_epochs=2, sgd_momentum=0.90, sgd_dampening=0.0, sgd_nesterov=True, sgd_weight_decay=1e-4, grad_clip_norm=1.0, ww_randomize=True, save_epoch_checkpoints=True)
SEEDS = tuple(DEFAULT_BASELINE_SEEDS); TARGETS = ('fc1.weight', 'fc2.weight')
GRID_EPOCHS = int(os.environ.get('RG_FML_GRID_EPOCHS', '5'))
GRID_BASE = replace(BASE, seed=int(SEEDS[0]), epochs=GRID_EPOCHS, save_epoch_checkpoints=False)
CANDIDATES = [FullMatrixLogConfig(mode='cone', momentum_projection='projected_state', normalization=normalization, projection_strength=strength, max_correction_ratio=0.10, apply_every_steps=cadence, warmup_steps=2*430, parameter_names=TARGETS) for normalization in ('full_m','self_consistent') for strength in (0.5,1.0) for cadence in (25,100)]
GRID_ROOT = RUN_ROOT/'validation_grid'; selected_path = GRID_ROOT/'selected_config.json'
if os.environ.get('RG_FML_SKIP_GRID','0') == '1':
    payload = json.loads(selected_path.read_text()); payload['parameter_names'] = tuple(payload['parameter_names']) if payload.get('parameter_names') else None
    BEST = FullMatrixLogConfig(**payload); GRID = pd.read_csv(GRID_ROOT/'grid_results_ranked.csv')
else:
    result = run_validation_grid(GRID_BASE, CANDIDATES, data_dir=DATA_DIR, output_dir=GRID_ROOT, device=DEVICE, progress=True, resume=True)
    BEST, GRID = result.selected_config, result.results
display(GRID); print('selected:', BEST)


In [ ]:
performance=[]; spectral=[]; corrections=[]
for seed in SEEDS:
    config = replace(BASE, seed=int(seed))
    control = run_mnist_sgd(config, rg_config=None, data_dir=DATA_DIR, output_dir=RUN_ROOT/'final'/'sgd_momentum'/f'seed_{seed}', device=DEVICE, evaluate_test=True, progress=True, resume=True)
    corrected = run_mnist_sgd(config, rg_config=BEST, data_dir=DATA_DIR, output_dir=RUN_ROOT/'final'/'full_matrix_log_rg_corrected'/f'seed_{seed}', device=DEVICE, evaluate_test=True, progress=True, resume=True)
    performance += [control.performance, corrected.performance]; spectral += [control.spectral, corrected.spectral]
    if not corrected.corrections.empty: corrections.append(corrected.corrections)
PERFORMANCE = pd.concat(performance, ignore_index=True); SPECTRAL = pd.concat(spectral, ignore_index=True, sort=False)
CORRECTIONS = pd.concat(corrections, ignore_index=True, sort=False) if corrections else pd.DataFrame()
PERFORMANCE_SUMMARY = summarize_numeric_metrics(PERFORMANCE, group_columns=('run','epoch'), metrics=('train_loss','validation_loss','test_loss','train_accuracy','validation_accuracy','test_accuracy'))
SPECTRAL_OK = SPECTRAL[SPECTRAL['status'].eq('ok')]
SPECTRAL_SUMMARY = summarize_numeric_metrics(SPECTRAL_OK, group_columns=('run','layer','epoch'), metrics=('alpha','ERG_gap','num_traps','m_midpoint'))
if not PERFORMANCE_SUMMARY['n'].eq(len(SEEDS)).all() or not SPECTRAL_SUMMARY['n'].eq(len(SEEDS)).all():
    raise RuntimeError('Final confidence intervals require all three complete runs')
AGG = RUN_ROOT/'final'/'aggregate'; AGG.mkdir(parents=True, exist_ok=True)
PERFORMANCE.to_csv(AGG/'performance_by_epoch_and_seed.csv', index=False); SPECTRAL.to_csv(AGG/'spectral_metrics_by_epoch_layer_and_seed.csv', index=False); CORRECTIONS.to_csv(AGG/'rg_corrections_by_step.csv', index=False)
PERFORMANCE_SUMMARY.to_csv(AGG/'performance_summary_95ci.csv', index=False); SPECTRAL_SUMMARY.to_csv(AGG/'spectral_summary_95ci.csv', index=False)
display(PERFORMANCE_SUMMARY); display(SPECTRAL_SUMMARY)
if not CORRECTIONS.empty:
    display(CORRECTIONS.groupby(['normalization','parameter']).agg(corrections=('correction_ratio','size'), mean_correction_ratio=('correction_ratio','mean'), projected_state_fraction=('momentum_state_projected','mean'), mean_kkt_residual=('active_set_kkt_residual','mean'), mean_violation_after=('max_signed_violation_after','mean')))
